# Part 2b: Quantization with Brevitas

In this notebook we train the same jet tagger architecture using [**Brevitas**](https://github.com/xilinx/brevitas) — AMD's/Xilinx's PyTorch-native quantization-aware training library — and convert it to an FPGA design via the **QONNX** (Quantized ONNX) frontend of hls4ml.

The workflow in this notebook is:
1. Define a Brevitas model with `QuantLinear` and `QuantReLU` layers and train it
2. Export to QONNX with `export_qonnx`
3. Clean up the QONNX graph and import it into hls4ml via `convert_from_onnx_model`

Make sure you have run `1_getting_started/1b_train_pytorch.ipynb` first so that the data and baseline PyTorch model are available.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('..')
import plotting

%matplotlib inline
seed = 0
np.random.seed(seed)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import brevitas.nn as qnn
from brevitas.export import export_qonnx

torch.manual_seed(seed)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

## Load the jet tagging dataset

We load the preprocessed arrays saved by `1b_train_pytorch.ipynb`.

In [ ]:
X_train_val = np.load('../data/jet-tagging/X_train_val.npy')
X_test = np.load('../data/jet-tagging/X_test.npy')
y_train_val = np.load('../data/jet-tagging/y_train_val.npy')
y_test = np.load('../data/jet-tagging/y_test.npy')
classes = np.load('../data/jet-tagging/classes.npy', allow_pickle=True)

## Define a Brevitas model

Brevitas replaces standard PyTorch layers with quantized equivalents:
- `QuantLinear` — linear layer with quantized weights (and optionally biases)
- `QuantReLU` — ReLU that clips and quantizes activations to a fixed bit-width

We use `weight_bit_width=6` and `bit_width=6` throughout, matching the 6-bit precision from Part 2a.
Unlike regular PyTorch, every forward pass applies the quantization, so the model trains with the actual fixed-point precisions used on the FPGA.

In [ ]:
class JetTaggerBrevitas(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = qnn.QuantLinear(16, 64, bias=False, weight_bit_width=6)
        self.relu1 = qnn.QuantReLU(bit_width=6)
        self.fc2 = qnn.QuantLinear(64, 32, bias=False, weight_bit_width=6)
        self.relu2 = qnn.QuantReLU(bit_width=6)
        self.fc3 = qnn.QuantLinear(32, 32, bias=False, weight_bit_width=6)
        self.relu3 = qnn.QuantReLU(bit_width=6)
        self.output = qnn.QuantLinear(32, 5, bias=False, weight_bit_width=6)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        x = self.output(x)
        return torch.softmax(x, dim=1)


model = JetTaggerBrevitas()
print(model)

## Train the model

Training is identical to standard PyTorch. Because softmax is the final operation, we use `NLLLoss` on the log of the model output.

In [ ]:
n_train = int(len(X_train_val) * 0.75)

X_tr = torch.FloatTensor(X_train_val[:n_train])
y_tr = torch.LongTensor(np.argmax(y_train_val[:n_train], axis=1))
X_val = torch.FloatTensor(X_train_val[n_train:])
y_val = torch.LongTensor(np.argmax(y_train_val[n_train:], axis=1))

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=1024, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.NLLLoss()

for epoch in range(20):
    model.train()
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = criterion(torch.log(model(X_batch).clamp(min=1e-7)), y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(torch.log(model(X_val).clamp(min=1e-7)), y_val).item()
    print(f'Epoch {epoch + 1:2d}  val_loss={val_loss:.4f}')

## Export to QONNX

Brevitas provides the `export_qonnx` function to serialize the trained model to an ONNX file that preserves the quantization (Quant nodes). This QONNX format is what hls4ml reads to infer per-layer precision.

After exporting, we run two cleanup steps:
1. `qonnx.util.cleanup` — removes redundant nodes and canonicalises the graph
2. `GemmToMatMul` — PyTorch exports linear layers as `Gemm` ops; hls4ml expects `MatMul`

In [ ]:
import qonnx.util.cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.gemm_to_matmul import GemmToMatMul

os.makedirs('../models', exist_ok=True)
raw_path = '../models/brevitas_model_part2b_raw.onnx'
final_path = '../models/brevitas_model_part2b.onnx'

model.eval()
export_qonnx(model, args=torch.randn(1, 16), export_path=raw_path, opset_version=14, dynamo=False)
print(f'Exported raw QONNX to {raw_path}')

# Clean and convert Gemm → MatMul
qonnx.util.cleanup.cleanup(raw_path, out_file=final_path)
model_wrapper = ModelWrapper(final_path)
model_wrapper = model_wrapper.transform(GemmToMatMul())
model_wrapper = qonnx.util.cleanup.cleanup_model(model_wrapper)
model_wrapper.save(final_path)
print(f'Cleaned QONNX saved to {final_path}')

## Check performance (float)

Before converting to HLS, let's check the trained model's accuracy in float simulation.

In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
with torch.no_grad():
    y_brevitas = model(torch.FloatTensor(X_test)).numpy()

print('Accuracy (Brevitas float): {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_brevitas, axis=1))))

## Convert to hls4ml via QONNX

`config_from_onnx_model` reads the Quant nodes embedded in the QONNX graph and derives per-layer precision automatically. 

In [ ]:
import hls4ml

config = hls4ml.utils.config_from_onnx_model(
    model_wrapper,
    granularity='name',
    backend='Vitis',
)

for layer in config.get('LayerName', {}):
    if layer.startswith('Softmax'):
        config['LayerName'][layer]['Implementation'] = 'legacy'

print('-----------------------------------')
plotting.print_dict(config)
print('-----------------------------------')

hls_model = hls4ml.converters.convert_from_onnx_model(
    model_wrapper,
    output_dir='../hls4ml_prjs/hls4ml_prj_brevitas_part2b',
    backend='Vitis',
    hls_config=config,
    part='xcu200-fsgd2104-2-e',
)
hls_model.compile()
y_hls = hls_model.predict(np.ascontiguousarray(X_test, dtype=np.float32))

## Compare

We compare the baseline PyTorch model (Part 1b), the Brevitas quantized model, and the hls4ml emulation.

In [ ]:
from sklearn.metrics import accuracy_score


# Load the baseline PyTorch model from Part 1b
class JetTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 32)
        self.output = nn.Linear(32, 5)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        return torch.softmax(self.output(x), dim=1)


model_ref = JetTagger()
model_ref.load_state_dict(torch.load('../models/pytorch_weights_part1.pt'))
model_ref.eval()
with torch.no_grad():
    y_ref = model_ref(torch.FloatTensor(X_test)).numpy()

print('Accuracy baseline (PyTorch):  {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_ref, axis=1))))
print('Accuracy Brevitas (float):    {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_brevitas, axis=1))))
print('Accuracy hls4ml:              {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_ref, list(classes))
plt.gca().set_prop_cycle(None)
_ = plotting.makeRoc(y_test, y_brevitas, list(classes), linestyle='--')
plt.gca().set_prop_cycle(None)
_ = plotting.makeRoc(y_test, y_hls, list(classes), linestyle=':')

from matplotlib.lines import Line2D
from matplotlib.legend import Legend

lines = [Line2D([0], [0], ls='-'), Line2D([0], [0], ls='--'), Line2D([0], [0], ls=':')]
leg = Legend(ax, lines, labels=['baseline (PyTorch)', 'Brevitas (float)', 'hls4ml'], loc='lower right', frameon=False)
ax.add_artist(leg)

## Synthesize

Now let's run Vitis HLS C-synthesis to get a first estimate of latency and resource usage.

**This can take several minutes.**


In [ ]:
hls_model.build(csim=False)

In [ ]:
hls4ml.report.read_vivado_report('../hls4ml_prjs/hls4ml_prj_brevitas_part2b')

Compare the DSP count above against the Part 1c baseline. With 6-bit quantization, every multiplication in this network is narrower than the ~10-bit threshold below which Vivado maps multiplications to LUT logic rather than DSP slices, significantly reducing the DSP usage.

## A note on HLS synthesis resource estimates

The resource numbers reported by Vitis HLS after HLS C-synthesis are **estimates** derived from the HLS's internal model and often do not truly reflect the final resource consumption of the model. These estimates **often overestimate** LUT consumption, sometimes by an order of magnitude.

For a more accurate picture of resource consumption, you should run **Vivado synthesis** (`vsynth`). This invokes the full Vivado synthesis flow on the generated RTL, producing estimates that are much closer to what you would see after implementation (place-and-route).

**This step can take 10–20 minutes.**

In [ ]:
hls_model.build(reset=False, csim=False, vsynth=True)

In [ ]:
hls4ml.report.read_vivado_report('../hls4ml_prjs/hls4ml_prj_brevitas_part2b')